# Week 2, day 5 (morning) — Worksheet 04 SOLUTIONS: while loops   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Several of these loops are deliberately written with a **safety counter** so
that a mistake prints something instead of hanging the kernel. That is not
ceremony — it is the habit that makes while loops safe to experiment with.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — While loops. Run this once.
balance = 1000.0          # Q3 halves this
funds = 1000.0            # Q5 spends this
withdrawals = [120.0, 300.0, 250.0, 400.0, 90.0]

job_queue = ["job-a", "job-b", "job-c"]

secret = 37
guesses = [50, 25, 30, 37]   # what a player happened to try, in order

print("balance:    ", balance)
print("withdrawals:", withdrawals)
print("job_queue:  ", job_queue)

PART A — The shape of a while loop, and the one line people forget

### Question 1

The counter loop, both ways. -> `0` to `4`, then `after the while loop, count is 5`; `0` to `4` again, then `after the for loop, i is 4`.

Same five numbers, and the loop variables end on **different values**.
`count` is 5 because the loop only stops once the condition is false, and
`count < 5` does not become false until count reaches 5. `i` is 4 because
a `for` loop stops when the collection runs out, and 4 was the last thing
in it.

That off-by-one is worth internalising. If you use the counter after the
loop — and people do, for a total or a row number — the `while` version
has already gone one past the last item it processed.

Three pieces make a while loop work: **initialise** before, **test** in the
condition, **change** in the body. The `for` loop gets all three for free,
which is exactly why you should prefer it whenever the count is known.

In [ ]:
count = 0
while count < 5:
    print(count)
    count = count + 1

print("after the while loop, count is", count)

for i in range(5):
    print(i)

print("after the for loop, i is", i)

### Question 2

The forgotten increment. -> `the body ran 20 times`, `count is still 0`, `count < 5 is still True`.

The body ran until the **guard** stopped it, not until the condition did.
`count` never moved, so `count < 5` was true on the twentieth pass exactly
as it was on the first, and without `guard < 20` this cell would still be
running now.

A runaway `while` in a notebook is not a crash — it is a hung kernel. The
`[*]` beside the cell never turns into a number, every cell after it queues
up behind it, and the fix is *Kernel → Interrupt*, or *Restart* if that
does not take.

While you are learning, put a guard in. It costs two lines and turns a
hang into a printout.

In [ ]:
count = 0
guard = 0

while count < 5 and guard < 20:
    guard = guard + 1     # the guard moves...
    # ...and count does not.

print("the body ran", guard, "times")
print("count is still", count)
print("count < 5 is still", count < 5)

PART B — When you cannot know the number of passes in advance

### Question 3

Halving until under 100. -> `62.5 after 4 halvings`.

1000 → 500 → 250 → 125 → 62.5. Four passes, and you could not have
written `range(4)` without doing that arithmetic first.

That is the whole case for `while`. A `for` loop needs its number of passes
decided before the first one; here the number of passes **is** the answer.
Anything shaped like *keep going until it converges / until the queue is
empty / until the API stops returning pages* is a while loop.

Note the loop stops **after** overshooting the threshold, not before —
125 was still ≥ 100, so it was halved again. The condition is checked
between passes, so the value you end with is always one step past the last
time the condition held.

In [ ]:
halvings = 0

while balance >= 100:
    balance = balance / 2
    halvings = halvings + 1

print(balance, "after", halvings, "halvings")

# A for loop needs to know how many passes there are BEFORE it starts. Here
# the number of halvings is the thing being calculated -- you would have to
# solve the problem to write range(...) correctly. That is what "indefinite"
# means, and it is the reason while exists.

### Question 4

Draining a queue. -> three `running job-x -- N left` lines counting 2, 1, 0, then `[] False`.

No counter, no `len()`, no index — the **list itself** is the condition.
This is worksheet 01's truthiness earning its place: a list with items in
it is true, an empty list is false, so `while job_queue:` reads as "while
there is work".

It is also self-correcting in a way a counter is not. If something else
adds a job mid-loop, this loop picks it up; a `for i in range(3)` would
not. That is the real difference between draining a queue and iterating a
list.

`.pop(0)` takes from the **front** and returns it, so the list shrinks on
every pass and the loop is guaranteed to end. (`.pop()` with no argument
takes from the back, which is faster but processes the queue in the wrong
order.)

In [ ]:
while job_queue:
    job = job_queue.pop(0)
    print("running", job, "--", len(job_queue), "left")

print(job_queue, bool(job_queue))

### Question 5

Withdrawals until one does not fit. -> three cleared (120, 300, 250), balance `330.0`, then `3 cleared, 2 not tried`.

The 400 does not fit in 330, so the condition goes false and the loop ends.
The arithmetic is right and the report is honest — as far as it goes.

**But the 90 would have fitted.** The loop stopped at the first failure
rather than skipping it and carrying on, so `2 not tried` is doing a lot of
work in that sentence. Whether that is correct depends entirely on what
the rule is meant to be: *process in order until one fails* and *process
every one that fits* are different policies with different answers, and
this code silently implements the first.

Ask which one you were asked for. "Skip the ones that do not fit" is
`continue`, and that is worksheet 05.

Also note the compound condition: `i < len(withdrawals)` **first**, then
the funds test. Reverse them and an empty-ish run would index off the end
of the list — Python evaluates `and` left to right and stops as soon as it
knows the answer, which is what makes the guard work.

In [ ]:
i = 0
cleared = 0

while i < len(withdrawals) and funds - withdrawals[i] >= 0:
    funds = funds - withdrawals[i]
    cleared = cleared + 1
    print("cleared", withdrawals[i], "-- balance now", funds)
    i = i + 1

print(cleared, "cleared,", len(withdrawals) - cleared, "not tried")
print("closing balance", funds)

PART C — while True, and how to get back out

### Question 6

The guessing game, no `input()`. -> `Guess lower...`, `Guess higher...`, `Guess higher...`, then `You successfully guessed 37 in 4 tries!`.

50 is too high, 25 and 30 too low, 37 right. `break` leaves the loop
immediately — the `i = i + 1` at the bottom never runs on that pass.

`while True:` says "the condition is not the thing that ends this loop".
Something inside has to, and here it is the `break` in the `else` branch.
Written as `while guess != secret:` you would need a guess before the loop
starts, which is exactly the awkwardness slide 75 works around with an
extra `input()` above the loop.

Compare the tries count with slide 75, which prints `4 tries` for four
guesses — the deck's version starts `tries = 1` and increments at the
bottom, so it happens to agree. Count what you actually mean to count.

In [ ]:
tries = 0
i = 0

while True:
    guess = guesses[i]
    tries = tries + 1
    if guess > secret:
        print("Guess lower...")
    elif guess < secret:
        print("Guess higher...")
    else:
        print(f"You successfully guessed {secret} in {tries} tries!")
        break
    i = i + 1

### Question 7

The same loop with a cap. -> `found: False -- after 3 tries`.

The secret is not in `bad_guesses`, so the success `break` never fires.
The loop still ends: it runs out of guesses, hits the length check, and
breaks with `found` still `False`.

Two separate protections are doing work here, and both are needed:
- `while tries < 10:` caps the total number of attempts;
- the `if tries >= len(...)` check stops `bad_guesses[tries]` from indexing
  off the end of the list.

And `found` is the part people leave out. Without a flag set inside the
loop, code after it cannot tell *why* the loop ended — success and
exhaustion look identical from the outside. Every retry loop needs to
report which one happened.

In [ ]:
bad_guesses = [50, 25, 30]
tries = 0
found = False

while tries < 10:
    if tries >= len(bad_guesses):
        break                      # ran out of guesses
    guess = bad_guesses[tries]
    tries = tries + 1
    if guess == secret:
        found = True
        break

print("found:", found, "-- after", tries, "tries")

PART D — Three ways a while loop goes wrong

### Question 8

Adding 0.1 until it equals 1.0. -> `4.999999999999998 after 50 passes`, then `x == 1.0 -> False` and `0.1 + 0.2 == 0.3 -> False`. **It ran the full 50 and never got there.**

Ten passes of `+ 0.1` do not produce `1.0`. They produce
`0.9999999999999999`, which is not equal to `1.0`, so the loop sailed past
and only the guard stopped it — at 50 passes and 4.999999999999998.

Binary floating point cannot represent 0.1 exactly, any more than decimal
can write 1/3 exactly. Every addition carries a little error and they
accumulate. `0.1 + 0.2 == 0.3` is `False` for the same reason, in every
language that uses IEEE 754 — this is not a Python quirk.

**Never test a float with `==` or `!=` in a loop condition.** Use `<` or
`>` (`while x < 1.0:`), count in integers and divide at the end, or use
`decimal.Decimal` for money. And note how this failed: not with an error,
but by running forty passes longer than it should have.

In [ ]:
x = 0.0
passes = 0

while x != 1.0 and passes < 50:
    x = x + 0.1
    passes = passes + 1

print(x, "after", passes, "passes")
print("x == 1.0 ->", x == 1.0)
print("0.1 + 0.2 == 0.3 ->", 0.1 + 0.2 == 0.3)

### Question 9

A condition that is false to begin with. -> `the body ran 0 times`, `funds is still 330.0`.

`while` is a **test-first** loop. The condition is checked before the first
pass, so if it starts false the body never runs at all — no error, no
output, nothing.

This is the same silence as worksheet 02's empty `range(5, 0)`, and it is
the first thing to check when a loop "does nothing". The loop is not
broken; the condition was never true.

(`funds` is 330.0 rather than 1000.0 because Q5 spent the rest of it. Run
the setup cell again if you want it back.)

Some languages have a do-while that always runs once. Python does not. If
you need at least one pass, use `while True:` with the test and a `break`
at the **bottom** of the body.

In [ ]:
runs = 0

while funds > 10000:
    runs = runs + 1
    funds = funds - 1

print("the body ran", runs, "times")
print("funds is still", funds)

### Question 10

Removing from a list while looping it with `for`. -> **`['b', 'd']`**, then `[]` from the while version.

Four items in, two left, nothing raised. The `for` loop walks by position:
it takes index 0 (`"a"`), you remove it, everything shifts left — so `"b"`
is now at index 0, and the loop moves on to index 1, which is `"c"`. `"b"`
is never visited. Same again for `"d"`.

This is the silent cousin of worksheet 02's Q10. A **dictionary** raises
`RuntimeError` when it changes size mid-loop; a **list** just skips half
the items and lets you carry on.

The `while` version is safe because it re-reads `jobs[0]` every pass — it
never holds a position across a change. Two other safe answers: loop over
a copy (`for job in jobs[:]`), or build a new list with a comprehension and
rebind. **Do not mutate the thing you are iterating.**

In [ ]:
jobs = ["a", "b", "c", "d"]

for job in jobs:
    jobs.remove(job)

print(jobs)      # not empty

jobs = ["a", "b", "c", "d"]
while jobs:
    jobs.remove(jobs[0])

print(jobs)      # empty

### Question 11

`while True` with no way out. -> `handling x`, `handling y`, then `IndexError: pop from empty list`.

Two passes work, the third has nothing left to pop, and the error is the
only reason this loop ever stopped. That is worth sitting with: without
the exception, this cell hangs the kernel forever.

So the error is the good outcome. It is the same shape as Q2's runaway
loop, except that `.pop(0)` happened to fail loudly rather than spin
quietly.

`while pending:` (Q4) never enters the body on an empty list. A capped
loop (Q7) stops after N attempts. Either fixes it, and both fix it *by
giving the loop a way to end* — which is what was actually missing.
`while True` is fine, but only when something inside it is guaranteed to
break.

In [ ]:
pending = ["x", "y"]

while True:
    item = pending.pop(0)
    print("handling", item)

# This is SUPPOSED to raise: IndexError, on the third pass, because pop(0)
# has nothing left to take.
#
# Q4's `while pending:` never enters the body on an empty list, and Q7's
# capped loop would have stopped after N attempts. Either one fixes this.
# `while True` with no exit inside it is the bug -- the break was missing,
# not the try/except.